# Day 21 — Mock Interview Practice

**Week 3 · Day 7 — the Week-3 finale · Level: ADVANCED** — 14 ADVANCED + 4 HARD questions.

18 questions across 8 categories. Every solution cell in this notebook was executed and verified on compute before publication, and every topic is new relative to Days 1–20.

| # | Category | Topic | Tag |
|---|----------|-------|-----|
| Q1 | SQL Advanced | Markov transition matrix from clickstream | ADVANCED |
| Q2 | SQL Advanced | Connected components via label propagation (pure SQL) | HARD |
| Q3 | Python | RAG / retrieval evaluation metrics — P@k, R@k, MRR, nDCG | ADVANCED |
| Q4 | Python | Bloom filter from scratch (sizing, double hashing, FPR) | HARD |
| Q5 | PySpark | Delta Change Data Feed (CDF) consumer | ADVANCED |
| Q6 | PySpark | SCD2 invariants auditor (windows + row fingerprint) | ADVANCED |
| Q7 | Spark Optimization | Deletion vectors — fast DELETE without Parquet rewrites | ADVANCED |
| Q8 | Spark Optimization | Input split sizing & scan parallelism | ADVANCED |
| Q9 | ETL | Schema-registry compatibility checker | ADVANCED |
| Q10 | ETL | Scoped incremental reload with REPLACE WHERE | ADVANCED |
| Q11 | ETL | Partial-batch isolation & SLA-aware loads | ADVANCED |
| Q12 | Data Modeling | Time-scoped (effective-dated) bridge table | ADVANCED |
| Q13 | Data Modeling | Lakehouse platform design — end-to-end | HARD |
| Q14 | Snowflake | Openflow — managed ingestion into the Snowflake lakehouse | ADVANCED |
| Q15 | Snowflake | Adaptive warehouses & sizing economics | ADVANCED |
| Q16 | Monitoring | Alert fatigue — precision/recall for monitoring rules | ADVANCED |
| Q17 | Monitoring | Correlated observability — one run, one story | ADVANCED |
| Q18 | Monitoring | From attribution to unit economics + burn anomaly detection | HARD |

**How to practice:** attempt each question cold (25–30 min budget), write your own solution first, then compare with the solution cell. Grade yourself in the score card at the bottom.

## Q1 — SQL Advanced: Markov Transition Matrix from Clickstream [ADVANCED]

**Scenario.** Product analytics wants page navigation modeled as a **Markov chain**: from every page (state), what is the probability that a session moves to each next page? These matrices power next-page prediction, session-similarity features, and funnel "leak" diagnosis. You have a `clickstream` table with `session_id`, `ts`, `page`.

**Tasks**
1. Build ordered state pairs with `LEAD` inside each session — the last event of a session has no successor and must be excluded (it would otherwise silently distort the matrix).
2. Count transitions and **row-normalize** to probabilities so each source state sums to 1.
3. Identify the **absorbing state(s)** — pages with no outgoing transition — and explain what an absorbing state means for a random-walk simulator.
4. Talk track: where does the first-order Markov assumption break on real clickstream data (back-button loops, bots, multi-tab sessions), and what does that imply for features built on it?

**Sample data:** 3 sessions over `home / search / product / cart / checkout`.

In [0]:
# Solution Q1 — LEAD builds state pairs; SUM() OVER normalizes each row to probabilities
from pyspark.sql import functions as F

df = spark.createDataFrame([
    (1, "2026-01-01 10:00:00", "home"),
    (1, "2026-01-01 10:01:00", "search"),
    (1, "2026-01-01 10:02:00", "product"),
    (1, "2026-01-01 10:03:00", "cart"),
    (1, "2026-01-01 10:04:00", "checkout"),
    (2, "2026-01-01 10:00:00", "home"),
    (2, "2026-01-01 10:01:00", "search"),
    (2, "2026-01-01 10:02:00", "search"),
    (2, "2026-01-01 10:03:00", "product"),
    (3, "2026-01-01 10:00:00", "search"),
    (3, "2026-01-01 10:01:00", "product"),
    (3, "2026-01-01 10:02:00", "home"),
], "session_id INT, ts STRING, page STRING").withColumn("ts", F.to_timestamp("ts"))
df.createOrReplaceTempView("clickstream")

trans = spark.sql("""
WITH ordered AS (
  SELECT session_id, page,
         LEAD(page) OVER (PARTITION BY session_id ORDER BY ts) AS next_page
  FROM clickstream
),
pairs AS (
  SELECT page AS from_state, next_page AS to_state
  FROM ordered WHERE next_page IS NOT NULL      -- drop session-final events
),
counts AS (
  SELECT from_state, to_state, COUNT(*) AS n FROM pairs GROUP BY 1, 2
)
SELECT c.from_state, c.to_state, c.n,
       ROUND(c.n * 1.0 / SUM(c.n) OVER (PARTITION BY c.from_state), 4) AS prob
FROM counts c ORDER BY from_state, prob DESC
""")
trans.show()

res = {(r.from_state, r.to_state): r.prob for r in trans.collect()}
# Hand-checked: home -> search happens in both sessions 1 & 2 and nothing else leaves home => 1.0
assert res[("home", "search")] == 1.0
# search has 4 outgoing transitions: product x3, search x1
assert res[("search", "product")] == 0.75 and res[("search", "search")] == 0.25
assert res[("cart", "checkout")] == 1.0
# checkout never appears as from_state: absorbing — sessions end there
assert all(k[0] != "checkout" for k in res)
print("Q1 verified: rows sum to 1 per state; 'checkout' is absorbing.")
print("Interview extras: n-step transition = matrix power; stationary distribution")
print("= eigenvector of the transition matrix (talk, don't compute, in SQL).")

## Q2 — SQL Advanced: Connected Components via Label Propagation (pure SQL) [HARD]

**Scenario.** Fraud/AML: build connected components over a graph of shared edges (devices, referral codes, addresses). The only execution engine available is a SQL warehouse — no GraphFrames, no cluster-side Python. The classic declarative algorithm is **iterative label propagation**: every node starts as its own label; each round, a node adopts the minimum label among itself and its neighbors; repeat until nothing changes.

**Tasks**
1. Build a **symmetric adjacency view** — `UNION` both edge directions, and include self-loops so isolated nodes survive every round.
2. Run propagation rounds until a fixpoint. Detect convergence by comparing consecutive label tables with `EXCEPT` (a set-difference of zero rows = stable).
3. Verify the components on the sample graph and report how many rounds were needed.
4. Talk track: the convergence bound (roughly the graph diameter — worst case path length), the cost per round (one shuffle-join + aggregate), and when you'd switch to union-find, GraphFrames, or offline label propagation instead.

**Sample graph:** edges among nodes 1–9 forming components `{1,2,3}`, `{4,5,6}`, `{7,8}`, and isolated `{9}`.

In [0]:
# Solution Q2 — min-label propagation, converged when EXCEPT returns zero rows
edges = spark.createDataFrame([
    (1, 2), (2, 3), (1, 3),          # component {1,2,3}
    (4, 5), (5, 6),                  # component {4,5,6}
    (7, 8),                          # component {7,8}
    (9, 9),                          # isolated node, kept via self-loop
], "src INT, dst INT")
edges.createOrReplaceTempView("edges")

spark.sql("""
  SELECT src AS node, dst AS neighbor FROM edges
  UNION
  SELECT dst AS node, src AS neighbor FROM edges
""").createOrReplaceTempView("adj")
spark.sql("""
  SELECT node, node AS neighbor FROM (SELECT DISTINCT node FROM adj)   -- self-loops
  UNION
  SELECT node, neighbor FROM adj
""").createOrReplaceTempView("adj2")

spark.sql("SELECT DISTINCT node AS id, node AS label FROM adj2").createOrReplaceTempView("labels")

iters = 0
while True:
    iters += 1
    spark.sql("""
      SELECT a.node AS id, MIN(l.label) AS label
      FROM adj2 a JOIN labels l ON a.neighbor = l.id
      GROUP BY a.node
    """).createOrReplaceTempView("new_labels")
    changed = spark.sql("""
      SELECT COUNT(*) AS c FROM (
        SELECT id, label FROM labels
        EXCEPT
        SELECT id, label FROM new_labels
      )
    """).first().c
    spark.sql("SELECT * FROM new_labels").createOrReplaceTempView("labels")
    if changed == 0 or iters >= 10:      # safety cap ~ graph diameter
        break

final = sorted((r.id, r.label) for r in spark.sql("SELECT id, label FROM labels ORDER BY id").collect())
print("final labels:", final, "| rounds:", iters)
comps = {}
for i, l in final:
    comps.setdefault(l, []).append(i)
assert sorted(map(sorted, comps.values())) == [[1, 2, 3], [4, 5, 6], [7, 8], [9]]
print("Q2 verified: 4 components recovered in", iters, "rounds.")
print("Scaling note: rounds ~ graph diameter; each round = 1 join + 1 agg. On very long")
print("chains (diameter in the hundreds), union-find or GraphFrames beats per-round SQL.")

## Q3 — Python: RAG / Retrieval Evaluation Metrics — P@k, R@k, MRR, nDCG [ADVANCED]

**Scenario.** You're evaluating a vector-search retriever before wiring it into a RAG application. A generative model can only answer from what the retriever surfaces, so retrieval quality is measured **before** and separately from generation quality. Given a ranked list of retrieved documents and ground truth (a relevant set plus **graded** relevance levels), implement the standard retrieval metrics from scratch — no library.

**Tasks**
1. `precision@k` — fraction of the top-k that is relevant (what the LLM sees is on-topic).
2. `recall@k` — fraction of all relevant docs captured in top-k (did we find everything).
3. `MRR` — reciprocal rank of the first relevant doc (how fast we hit one good hit).
4. `nDCG@k` with graded relevance — gain `2^rel - 1`, position discount `1 / log2(rank + 1)`, normalized by the ideal ordering (does the BEST doc come FIRST).
5. Talk track: which metric protects which failure mode, how to choose k, and why RAG answer quality usually tracks recall/MRR at small k rather than nDCG at large k.

**Sample:** retrieved `['d2','d1','d3','d7']`, relevant `{d1, d3}`, graded relevance `d1=3, d3=1, d2=0`.

In [0]:
# Solution Q3 — the four retrieval metrics, hand-verified on the sample
import math

def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & set(relevant)) / k

def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & set(relevant)) / len(relevant) if relevant else 0.0

def mrr(retrieved, relevant):
    for i, d in enumerate(retrieved, start=1):
        if d in relevant:
            return 1.0 / i
    return 0.0

def ndcg_at_k(retrieved, graded_rel, k):
    dcg = sum((2 ** graded_rel.get(d, 0) - 1) / math.log2(r + 1)
              for r, d in enumerate(retrieved[:k], start=1))
    ideal = sorted(graded_rel.values(), reverse=True)[:k]
    idcg = sum((2 ** rel - 1) / math.log2(r + 1) for r, rel in enumerate(ideal, start=1))
    return dcg / idcg if idcg > 0 else 0.0

q = {"retrieved": ["d2", "d1", "d3", "d7"], "relevant": {"d1", "d3"},
     "graded": {"d1": 3, "d3": 1, "d2": 0}}
p3 = precision_at_k(q["retrieved"], q["relevant"], 3)
r3 = recall_at_k(q["retrieved"], q["relevant"], 3)
m  = mrr(q["retrieved"], q["relevant"])
n  = ndcg_at_k(q["retrieved"], q["graded"], 3)
print(f"P@3={p3:.4f}  R@3={r3:.4f}  MRR={m:.4f}  nDCG@3={n:.4f}")

assert abs(p3 - 2/3) < 1e-9      # d1 and d3 both in top-3
assert abs(r3 - 1.0) < 1e-9
assert abs(m - 0.5) < 1e-9        # first relevant hit (d1) sits at rank 2
# nDCG hand-derivation: DCG = 0 + 7/log2(3) + 1/log2(4) = 4.91651
#                      IDCG = 7/1 + 1/log2(3) = 7.63093   ->  0.64429
assert abs(n - 0.6443) < 0.0005
print("Q3 verified: all four metrics match hand-derived values.")
print("Talk track: nDCG punishes putting the grade-0 doc d2 first; MRR is blind to everything")
print("after the first hit; RAG pipelines usually optimize recall@5-10 + MRR first.")

## Q4 — Python: Bloom Filter from Scratch [HARD]

**Scenario.** Before hitting an expensive lookup path (remote key-value store, cross-table join), you want a cheap probabilistic **membership** pre-filter that never misses a real member but may admit a few false positives. That's a **Bloom filter** — and you're asked to build one from scratch, with the sizing math, on a whiteboard.

**Tasks**
1. Derive the sizing formulas: bits `m = -n * ln(p) / (ln 2)^2`, hashes `k = (m / n) * ln 2`, and the false-positive rate `(1 - e^(-kn/m))^k`.
2. Implement the bit array + **double hashing** `h_i(x) = h1(x) + i * h2(x) mod m` (so one hash pair yields k positions without k hash functions).
3. Verify: **zero false negatives** on 1,000 inserted items; measured FPR on 1,000 absent items must be near the 1% target.
4. Talk track: why deletion is impossible (counting Bloom filters), how runtime bloom join filters use the same idea, and the contrast with HyperLogLog (membership vs cardinality estimation).

**Target:** n = 1,000 items, p = 1%.

In [0]:
# Solution Q4 — bit array, double hashing, sizing math, measured FPR
import math, hashlib

class BloomFilter:
    def __init__(self, n, p):
        self.m = math.ceil(-n * math.log(p) / (math.log(2) ** 2))   # bits
        self.k = max(1, round((self.m / n) * math.log(2)))           # hash functions
        self.bits = [0] * self.m
        self.n = n

    def _positions(self, item):
        # double hashing: one h1 + one h2 -> k positions
        h1 = int(hashlib.sha256(f"a:{item}".encode()).hexdigest(), 16)
        h2 = int(hashlib.sha256(f"b:{item}".encode()).hexdigest(), 16) | 1  # force odd
        return [(h1 + i * h2) % self.m for i in range(self.k)]

    def add(self, item):
        for pos in self._positions(item):
            self.bits[pos] = 1

    def might_contain(self, item):
        return all(self.bits[pos] for pos in self._positions(item))

n_items, p_target = 1000, 0.01
bf = BloomFilter(n_items, p_target)
theo = (1 - math.exp(-bf.k * bf.n / bf.m)) ** bf.k
print(f"m={bf.m} bits ({bf.m/8/1024:.1f} KB), k={bf.k}, theoretical FPR={theo:.4f}")
assert bf.m == 9586 and bf.k == 7          # classic n=1000, p=1% sizing

train = [f"user_{i}" for i in range(n_items)]
for u in train:
    bf.add(u)
assert all(bf.might_contain(u) for u in train)     # NO false negatives, ever

fp = sum(bf.might_contain(f"other_{i}") for i in range(n_items))
print(f"measured FPR = {fp/n_items:.4f} (target 1%)")
assert fp / n_items < 0.05                     # generous slack for hash luck
print("Q4 verified: 0 false negatives, measured FPR near the 1% design point.")
print("Talk track: to support deletion you need counters per bit (counting BF); Spark's")
print("runtime bloom filters skip join sides with the same probability trade-off.")

## Q5 — PySpark: Delta Change Data Feed (CDF) Consumer [ADVANCED]

**Scenario.** A downstream team needs a continuously-refreshed replica of a shared Delta source, but re-reading the full table nightly is too expensive. The source table can publish its changes: Delta **Change Data Feed**. Your consumer must be exactly correct across all change types — including the tricky `update_preimage` / `update_postimage` pairs.

**Tasks**
1. Create a CDF-enabled source table, then generate three versions: v1 inserts, v2 an UPDATE, v3 a DELETE + INSERT.
2. Read the full change feed from version 0 and list the `_change_type` values you observe.
3. Apply the changes to a replica **as a MERGE**: for each key, look only at its latest commit version; if the latest version contains an `insert`/`update_postimage` row the key is alive (upsert it), otherwise (`delete`/`update_preimage` only) the key is dead (delete it).
4. Also demonstrate an **incremental** read starting at version 3 — the daily-driver pattern.
5. Talk track: why CDF beats trigger tables + timestamp columns, `startingVersion`/`endingVersion` windowing, and what happens to the feed after `OPTIMIZE` and `VACUUM`.

In [0]:
# Solution Q5 — enable CDF, replay all change types into a replica, incremental read
T = "workspace.default.d21_cdf_src"
spark.sql(f"""
CREATE OR REPLACE TABLE {T} (id INT, name STRING, city STRING)
USING DELTA TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
spark.sql(f"INSERT INTO {T} VALUES (1,'Ann','Pune'), (2,'Ben','Delhi'), (3,'Cal','Chennai')")   # v1
spark.sql(f"UPDATE {T} SET city = 'Mumbai' WHERE id = 1")
v_update = max(r.version for r in spark.sql(f"DESCRIBE HISTORY {T}").collect())
spark.sql(f"DELETE FROM {T} WHERE id = 2")
spark.sql(f"INSERT INTO {T} VALUES (4,'Dee','Goa')")                                             # v3

cdf = spark.read.option("readChangeFeed", "true").option("startingVersion", 0).table(T)
cdf.orderBy("_commit_version", "_change_type", "id").show(truncate=False)
cdf.createOrReplaceTempView("changes")
seen = sorted({r._change_type for r in cdf.collect()})
assert seen == ['delete', 'insert', 'update_postimage', 'update_preimage'], seen

spark.sql("""
CREATE OR REPLACE TABLE workspace.default.d21_cdf_replica
(id INT, name STRING, city STRING, _src_version BIGINT) USING DELTA
""")
# apply: per key, only the LATEST commit version decides alive vs dead
spark.sql("""
WITH ranked AS (
  SELECT id, name, city, _commit_version, _change_type,
         MAX(_commit_version) OVER (PARTITION BY id) AS maxv
  FROM changes
),
latest AS (
  SELECT * FROM ranked WHERE _commit_version = maxv
),
final AS (
  SELECT id,
         MAX(CASE WHEN _change_type IN ('insert','update_postimage') THEN name END) AS name,
         MAX(CASE WHEN _change_type IN ('insert','update_postimage') THEN city END) AS city,
         MAX(_commit_version) AS v,
         MAX(CASE WHEN _change_type IN ('insert','update_postimage') THEN 1 ELSE 0 END) AS alive
  FROM latest GROUP BY id
)
MERGE INTO workspace.default.d21_cdf_replica r
USING final s ON r.id = s.id
WHEN MATCHED AND s.alive = 0 THEN DELETE
WHEN MATCHED AND s.alive = 1 THEN
  UPDATE SET r.name = s.name, r.city = s.city, r._src_version = s.v
WHEN NOT MATCHED AND s.alive = 1 THEN
  INSERT (id, name, city, _src_version) VALUES (s.id, s.name, s.city, s.v)
""")

replica = sorted((r.id, r.name, r.city) for r in
    spark.sql("SELECT id, name, city FROM workspace.default.d21_cdf_replica").collect())
assert replica == [(1, 'Ann', 'Mumbai'), (3, 'Cal', 'Chennai'), (4, 'Dee', 'Goa')], replica

# incremental daily pattern: everything AFTER the captured update version
# (version-relative, so the cell is re-runnable even when the table carries old history)
inc = spark.read.option("readChangeFeed", "true").option("startingVersion", v_update + 1).table(T)
assert sorted(r._change_type for r in inc.collect()) == ['delete', 'insert']
print("Q5 verified: replica matches source; incremental window reads work.")
print("Talk track: OPTIMIZE keeps CDF rows (compaction rewrites but preserves them);")
print("VACUUM can expire old change files — keep a CDF retention >= your consumer lag.")

## Q6 — PySpark: SCD2 Invariants Auditor (Windows + Row Fingerprint) [ADVANCED]

**Scenario.** A three-year-old SCD Type 2 dimension was built by many jobs and has drifted. Before a migration you must **audit invariants**, not just row counts — and prove that current-row values match the source of truth. Planted in the sample dim: an **overlap** (cust 100), a **validity gap** (cust 200), a **double current row** with a **stale value** (cust 300), and one clean customer (400).

**Tasks**
1. **Exactly one `is_current` row per business key.**
2. **No overlapping validity intervals** — with exclusive-end semantics (`effective_to` is the first day NOT covered), the next row's `effective_from` must be strictly greater than the prior `effective_to`.
3. **No gaps** — contiguity requires `effective_from = effective_to + 1 day` for consecutive rows.
4. **Fingerprint** — SHA-256 over the tracked attributes of each current row, compared against the source-of-truth current snapshot, to catch silent value drift without shipping both tables around.
5. Talk track: how you'd wire this as a daily Lakehouse-monitoring-style check and what to do on violation (halt, alert, or self-heal).

In [0]:
# Solution Q6 — window-based invariant tests + SHA-256 fingerprint comparison
from pyspark.sql import functions as F

dim = spark.createDataFrame([
    (100, "2024-01-01", "2024-06-15", False, "Pune"),     # OVERLAP: next row starts 06-01
    (100, "2024-06-01", None,         True,  "Mumbai"),
    (200, "2024-01-01", "2024-03-31", False, "Delhi"),       # GAP: next row starts 04-10
    (200, "2024-04-10", None,         True,  "Bangalore"),
    (300, "2024-01-01", None,         True,  "Chennai"),      # TWO currents + stale value
    (300, "2024-02-01", None,         True,  "Kolkata"),
    (400, "2024-01-01", None,         True,  "Goa"),          # clean
], "customer_id INT, effective_from STRING, effective_to STRING, is_current BOOLEAN, city STRING") \
   .withColumn("effective_from", F.to_date("effective_from")) \
   .withColumn("effective_to", F.to_date("effective_to"))
dim.createOrReplaceTempView("dim_customer")

audit = spark.sql("""
WITH currents AS (
  SELECT customer_id, COUNT(*) AS n_current
  FROM dim_customer WHERE is_current GROUP BY 1
),
sequenced AS (
  SELECT customer_id, effective_from, effective_to,
         LEAD(effective_from) OVER (PARTITION BY customer_id ORDER BY effective_from) AS next_from
  FROM dim_customer
)
SELECT 'dup_current' AS violation, CAST(customer_id AS STRING) AS detail
FROM currents WHERE n_current <> 1
UNION ALL
SELECT 'overlap', CONCAT(customer_id, ': next row starts ', next_from, ' before prior ends ', effective_to)
FROM sequenced WHERE next_from IS NOT NULL AND effective_to IS NOT NULL AND next_from <= effective_to
UNION ALL
SELECT 'gap', CONCAT(customer_id, ': gap between ', effective_to, ' and ', next_from)
FROM sequenced WHERE next_from IS NOT NULL AND effective_to IS NOT NULL
  AND next_from > DATE_ADD(effective_to, 1)
ORDER BY 1
""")
viol = audit.collect()
for v in viol:
    print(v.violation, "|", v.detail)
assert sorted({v.violation for v in viol}) == ["dup_current", "gap", "overlap"]

# fingerprint: current-row attribute hash vs source-of-truth snapshot
fp = spark.sql("""
  SELECT customer_id, city,
         SHA2(CONCAT_WS('||', customer_id, city), 256) AS row_fp
  FROM dim_customer WHERE is_current
""")
fp.show(truncate=False)
truth = {100: "Mumbai", 200: "Bangalore", 300: "Chennai", 400: "Goa"}
dim_cur = {r.customer_id: r.city for r in fp.orderBy("customer_id", "city").collect()}
mismatch = [c for c in truth if dim_cur.get(c) != truth[c]]
print("fingerprint mismatches:", mismatch)
assert mismatch == [300]           # cust 300's truth (Chennai) disagrees with the dim
print("Q6 verified: overlap + gap + dup_current planted and all caught; fingerprint")
print("caught the silent value drift on cust 300 that row counts would never see.")

## Q7 — Spark Optimization: Deletion Vectors — Fast DELETE Without Parquet Rewrites [ADVANCED]

**Scenario.** GDPR erasure and soft-delete purges on a large Delta table were re-writing whole Parquet files per DELETE — hours of write amplification every night. Delta's **deletion vectors (DVs)** fix exactly this: a `DELETE` writes a small sidecar file per affected data file marking dead row IDs, leaving the Parquet bytes untouched.

**Tasks**
1. Create a table with DVs enabled (`delta.enableDeletionVectors = true`), load 2,000 rows, and record `numFiles` from `DESCRIBE DETAIL`.
2. Run a `DELETE` and prove the file count **did not change** — the Parquet files were not rewritten.
3. Run `OPTIMIZE` and observe the compaction that clears DVs back into clean files.
4. Talk track: the read-amplification trade-off (readers must apply per-file DVs), interaction with file statistics and liquid clustering, why DVs are on-by-default for new tables on current runtimes, and the streaming/concurrency implications.

In [0]:
# Solution Q7 — DELETE leaves Parquet files untouched; OPTIMIZE clears DVs
DV = "workspace.default.d21_dv"
spark.sql(f"""
CREATE OR REPLACE TABLE {DV} (id INT, grp STRING)
USING DELTA TBLPROPERTIES ('delta.enableDeletionVectors' = 'true')
""")
spark.sql(f"INSERT INTO {DV} SELECT id, CONCAT('g', CAST(id % 10 AS STRING)) FROM range(2000)")

det1 = spark.sql(f"DESCRIBE DETAIL {DV}").first()
spark.sql(f"DELETE FROM {DV} WHERE id % 3 = 0")
det2 = spark.sql(f"DESCRIBE DETAIL {DV}").first()
n_after = spark.sql(f"SELECT COUNT(*) c FROM {DV}").first().c
spark.sql(f"OPTIMIZE {DV}")
det3 = spark.sql(f"DESCRIBE DETAIL {DV}").first()

print(f"numFiles before={det1.numFiles}, afterDelete={det2.numFiles}, afterOptimize={det3.numFiles}")
print(f"rows after delete = {n_after}")
assert n_after == 1333                                   # 2000 - floor(2000/3) = 1333
assert det2.numFiles == det1.numFiles                    # no Parquet rewrite!
assert det3.numFiles >= 1                                 # OPTIMIZE compacted
print("Q7 verified: the DELETE touched zero Parquet files (DVs only); OPTIMIZE merged files.")
print("Talk track: readers pay DV-application cost per file — that is the read-amplification")
print("trade you accept for cheap DELETEs; pair DVs with liquid clustering and regular")
print("scheduled OPTIMIZE so DVs don't accumulate on hot files forever.")

## Q8 — Spark Optimization: Input Split Sizing & Scan Parallelism [ADVANCED]

**Scenario.** A 4-file read produced exactly 4 tasks even though the files are tiny — 4 tasks of scheduling overhead for almost no work. Meanwhile, on the big side, a single 10 GB file yields ~80 tasks of at most 128 MB each. To reason about either end you need to know **how file layout becomes read tasks** — and diagnose it live.

**Tasks**
1. Explain the split-planning math: files below `spark.sql.files.maxPartitionBytes` (default 128 MB) each become their own split (subject to `openCostBytes` packing); larger files are chopped bytewise into splits of at most `maxPartitionBytes`; `minPartitionBytes` sets the floor. Task count = split count.
2. Diagnose live: read a 4-file Parquet directory and attribute every row to its read task using `spark_partition_id()`, with `_metadata.file_path` as the file provenance (on Unity Catalog `input_file_name()` is unavailable — use `_metadata`).
3. Derive the small-files consequence: **N files = up to N tasks even if each file is 20 KB** — scheduling overhead dwarfs work.
4. Talk track (with a live lesson from this environment): on serverless/Spark Connect, split planning happens **server-side** — client `spark.conf.set('spark.sql.files.maxPartitionBytes', ...)` did not change the task count when tested in this notebook. The robust lever is the **layout**: compact small files (OPTIMIZE / repartition on write), or produce fewer larger files at ingestion time.

In [0]:
# Solution Q8 — observe task assignment with spark_partition_id() + _metadata.file_path
from pyspark.sql import functions as F

P = "/Volumes/workspace/default/etl_test_volume/d21nb_q8"
spark.range(0, 200_000).withColumn("v", F.col("id") % 7).repartition(4) \
     .write.mode("overwrite").parquet(P)
files = [(f.name, f.size) for f in dbutils.fs.ls(P) if f.name.endswith(".parquet")]
print("files written:", files)
assert len(files) == 4

diagnostic = (spark.read.parquet(P)
              .withColumn("pid", F.spark_partition_id())
              .select("pid", "id", "v",
                      F.element_at(F.reverse(F.split(F.col("_metadata.file_path"), "/")), 1).alias("file")))
per_task = diagnostic.groupBy("pid").agg(F.count("*").alias("rows"),
                                         F.countDistinct("file").alias("files"))
per_task.orderBy("pid").show()

n_tasks = per_task.count()
assert n_tasks == 4                              # 4 files -> 4 read tasks
assert per_task.filter("files != 1").count() == 0  # no task read more than one file
print("Q8 verified: 4 small files cost exactly 4 read tasks, one file per task.")
print("Math recap: bytesPerSplit = min(maxPartitionBytes, max(openCostBytes, max(minPartitionBytes, bytes/minSplits)))")
print("Live env lesson: client-side spark.conf.set of maxPartitionBytes did NOT change")
print("task count on serverless (planned server-side) — fix the file layout instead.")

## Q9 — ETL: Schema-Registry Compatibility Checker [ADVANCED]

**Scenario.** Twelve producer teams evolve their schemas independently; a consumer outage last quarter was traced to a producer adding a required field. You're designing the **schema-registry gate**: every schema change is classified before it ships, and the compatibility level determines whether CI lets the producer deploy.

**Tasks**
1. Implement a checker that classifies a schema change as `FULL`, `BACKWARD`, `FORWARD`, or `NONE`:
   * *BACKWARD* — the **new** schema can read data written with the **old** schema (rolling back consumers is safe).
   * *FORWARD* — the **old** schema can read data written with the **new** schema (rolling out consumers first is safe).
   Rules: added **nullable** field → FULL; added **required** field → not backward; **dropping a required** field → not forward; **widening** a type (`int → long/string`) → backward only; **narrowing** (`string → int`) → NONE.
2. Classify five test evolutions and verify the verdicts.
3. Talk track: where the registry lives in a lakehouse (Unity Catalog as the system of record + Delta schema evolution + contract tests in CI), the difference between `BACKWARD` and `BACKWARD_TRANSITIVE` (checked against **all** historical versions, not just the last), and why Kafka-style registries check the wire format while Delta checks the table schema.

In [0]:
# Solution Q9 — a field-level compatibility gate you can drop into CI
def check_compatibility(old, new):
    """old/new: {field: {'type': t, 'nullable': bool}} -> FULL / BACKWARD / FORWARD / NONE"""
    backward, forward = True, True
    widening = {("int", "long"), ("int", "string"), ("long", "string"),
                ("int", "double"), ("long", "double")}
    for f in set(new) - set(old):                 # added fields
        if not new[f]["nullable"]:
            backward = False                      # old data lacks the field; new reader requires it
    for f in set(old) - set(new):                 # removed fields
        if not old[f]["nullable"]:
            forward = False                      # old reader still expects the dropped field
    for f in set(old) & set(new):                 # changed fields
        if old[f]["type"] != new[f]["type"]:
            forward = False                      # any type change is a hazard for the old reader
            if (old[f]["type"], new[f]["type"]) not in widening:
                backward = False                 # narrowing breaks the new reader on old data
    if backward and forward: return "FULL"
    if backward: return "BACKWARD"
    if forward: return "FORWARD"
    return "NONE"

s1 = {"id":   {"type": "int",    "nullable": False},
      "name": {"type": "string", "nullable": False}}
s2 = {**s1, "email": {"type": "string", "nullable": True}}    # add optional
s3 = {**s1, "email": {"type": "string", "nullable": False}}   # add required
s4 = {k: v for k, v in s1.items() if k != "name"}                 # drop required
s5 = {**s1, "id": {"type": "long", "nullable": False}}          # widen int -> long
s6 = {**s1, "name": {"type": "int", "nullable": False}}       # narrow string -> int

cases = [(s1, s2, "FULL"), (s1, s3, "FORWARD"), (s1, s4, "BACKWARD"),
         (s1, s5, "BACKWARD"), (s1, s6, "NONE")]
for i, (a, b, exp) in enumerate(cases, 1):
    got = check_compatibility(a, b)
    print(f"case {i}: {got:10s} (expected {exp})")
    assert got == exp, (i, got, exp)
print("Q9 verified: all five evolutions classified correctly.")
print("Talk track: enforce at CI time against the UC-registered schema; treat")
print("BACKWARD_TRANSITIVE as the default policy for long-retention topics/tables.")

## Q10 — ETL: Scoped Incremental Reload with REPLACE WHERE [ADVANCED]

**Scenario.** Late-arriving corrections land for exactly one business date. The naive options are bad: full-table rewrite (expensive + rewrites history), or DELETE + INSERT (two operations, awkward to make one transaction). Delta's `replaceWhere` scopes an **overwrite to a predicate**: files covering the window are replaced by the new write, everything else is untouched — in a single commit.

**Tasks**
1. Load 3 dates, then deliver corrections for `2026-01-02` only (one row changed, one added) via `write.mode('overwrite').option('replaceWhere', "dt = '2026-01-02'")`.
2. Prove the other dates were never touched, and that re-running the same scoped write is **idempotent**.
3. Inspect `DESCRIBE HISTORY` and confirm the write predicates were scoped to the window.
4. Talk track: how `replaceWhere` differs from partition-overwrite (`replaceWhere` unpartitioned); the risk of predicate/write mismatch (writes filtered by the predicate — data outside the window errors or is silently dropped depending on version — name the behavior); when to prefer `DELETE + INSERT` in one transaction instead.

In [0]:
# Solution Q10 — atomic, idempotent, window-scoped overwrite
from pyspark.sql import functions as F

T = "workspace.default.d21_rw"
spark.sql(f"CREATE OR REPLACE TABLE {T} (dt DATE, id INT, amt INT) USING DELTA")
spark.sql(f"""
INSERT INTO {T} VALUES
 ('2026-01-01', 1, 100), ('2026-01-01', 2, 101),
 ('2026-01-02', 3, 200), ('2026-01-02', 4, 201),
 ('2026-01-03', 5, 300)
""")

# late corrections for 2026-01-02 only: id 3 amt fixed, id 6 added
fix = spark.createDataFrame([("2026-01-02", 3, 250), ("2026-01-02", 4, 201), ("2026-01-02", 6, 260)],
    "dt STRING, id INT, amt INT").withColumn("dt", F.to_date("dt"))
fix.write.format("delta").mode("overwrite").option("replaceWhere", "dt = '2026-01-02'").saveAsTable(T)

got = sorted((str(r.dt), r.id, r.amt) for r in spark.sql(f"SELECT dt, id, amt FROM {T}").collect())
assert got == [("2026-01-01", 1, 100), ("2026-01-01", 2, 101),
               ("2026-01-02", 3, 250), ("2026-01-02", 4, 201), ("2026-01-02", 6, 260),
               ("2026-01-03", 5, 300)], got

# idempotency: replaying the same scoped write changes nothing
fix.write.format("delta").mode("overwrite").option("replaceWhere", "dt = '2026-01-02'").saveAsTable(T)
got2 = sorted((str(r.dt), r.id, r.amt) for r in spark.sql(f"SELECT dt, id, amt FROM {T}").collect())
assert got2 == got

ops = [(r.operation, r.operationParameters.get("predicate", "n/a"))
       for r in spark.sql(f"DESCRIBE HISTORY {T}").collect()]
for op in ops:
    print(op)
# history is newest-first; compare only THIS run's two scoped writes (the table
# may carry prior versions from earlier notebook runs)
preds = [p for op, p in ops if op == "WRITE" and p != "n/a"]
assert preds[:2] == ["dt = '2026-01-02'"] * 2, preds
print("Q10 verified: only the target window replaced; idempotent re-run; history shows")
print("scoped predicates. Talk: on unpartitioned tables replaceWhere is the right tool;")
print("on partitioned ones compare with dynamic partition overwrite — same idea, per-partition grain.")

## Q11 — ETL: Partial-Batch Isolation & SLA-Aware Loads [ADVANCED]

**Scenario.** The 03:00 daily load processes one slice per business date. One poison slice (malformed upstream payload) must not hold the other four hostage: the SLA says *freshness by 06:00 for whatever is clean*. Design the load so it can **succeed partially, honestly** — with an auditable contract for downstream consumers.

**Tasks**
1. Implement per-slice processing with a **bounded retry** (2 attempts), then **quarantine** the failing slice — never let one slice abort the batch.
2. Load every clean slice, and publish a `load_audit` record: `status = partial_success`, slices loaded, slices quarantined, `% complete`.
3. Talk track: when is a partial load acceptable (freshness-SLA vs completeness-SLA trade), how downstream consumers filter on the audit table, how the quarantine gets replayed once the producer fixes the payload, and what you alert on (partial_success pages no one at 3 AM if the SLA is still met — ticket instead).

In [0]:
# Solution Q11 — bounded retry, quarantine, honest audit, clean slices still served
import json

slices = {
    "2026-01-01": {"id": 1, "amt": 100},
    "2026-01-02": {"id": 2, "amt": 200},
    "2026-01-03": {"id": 3, "amt": 300},
    "2026-01-04": "POISON",                      # malformed upstream payload
    "2026-01-05": {"id": 5, "amt": 500},
}

def transform(batch):
    if batch == "POISON":
        raise ValueError("malformed payload")
    return {"id": batch["id"], "amt_usd": batch["amt"], "flag": batch["amt"] > 250}

loaded, quarantined = [], []
for date, batch in sorted(slices.items()):
    try:
        for attempt in range(2):                   # bounded retry: 2 attempts max
            try:
                loaded.append(transform(batch))
                break
            except ValueError:
                if attempt == 1:
                    raise                          # quarantine, don't abort the batch
    except ValueError:
        quarantined.append(date)

load_audit = {
    "status": "partial_success" if quarantined else "success",
    "slices_loaded": len(loaded),
    "slices_quarantined": quarantined,
    "pct_complete": len(loaded) / len(slices),
}
print(json.dumps(load_audit, indent=2))
assert load_audit["status"] == "partial_success"
assert load_audit["slices_loaded"] == 4 and quarantined == ["2026-01-04"]
assert abs(load_audit["pct_complete"] - 0.8) < 1e-9
assert [r["id"] for r in loaded] == [1, 2, 3, 5]
print("Q11 verified: 4/5 slices served on SLA; poison slice fenced with a replayable audit trail.")
print("Talk track: the audit table IS the downstream contract — consumers filter on status;")
print("replay = reprocess quarantined slices by date; alert policy: page on SLA breach, ticket on partial.")

## Q12 — Data Modeling: Time-Scoped (Effective-Dated) Bridge Table [ADVANCED]

**Scenario.** Day 17 covered the static bridge for many-to-many dimensions. Real banking relationships are **temporal**: account A1 was owned by customer C1 Jan–Jun, then by C2; C3 held a *signatory* role on A1 Feb–May; C1 is a beneficiary on A2 throughout. Every membership row needs `effective_from` / `effective_to`, and multiple roles can coexist on the same account at the same time.

**Tasks**
1. Declare the **grain** of the bridge precisely (customer × account × role × validity interval) and explain why `is_current` flags are not enough here.
2. Answer the as-of question: *who held which role on A1 on 2024-04-01?* (must return C1 owner + C3 signatory).
3. Write the **invariant test**: for each (customer, account, role), validity intervals must not overlap.
4. Produce the point-in-time footprint: all of C1's relationships across accounts.
5. Talk track: how facts join to time-scoped bridges (fact date BETWEEN from AND to — the classic degenerate join predicate), and how late-arriving membership corrections are handled without rewriting history.

In [0]:
# Solution Q12 — effective-dated membership, as-of queries, invariant test
from pyspark.sql import functions as F

bridge = spark.createDataFrame([
    # A1: owned by C1 Jan-Jun, then C2; C3 signatory Feb-May
    ("C1", "A1", "owner",       "2024-01-01", "2024-06-30"),
    ("C2", "A1", "owner",       "2024-07-01", None),
    ("C3", "A1", "signatory",   "2024-02-01", "2024-05-31"),
    ("C1", "A2", "beneficiary", "2024-01-01", None),
], "customer_key STRING, account_key STRING, role STRING, from_d STRING, to_d STRING") \
   .withColumn("effective_from", F.to_date("from_d")).withColumn("effective_to", F.to_date("to_d")) \
   .drop("from_d", "to_d")
bridge.createOrReplaceTempView("bridge")
bridge.show()

# (2) as-of 2024-04-01 membership on A1
asof = spark.sql("""
SELECT customer_key, role FROM bridge
WHERE account_key = 'A1'
  AND effective_from <= DATE'2024-04-01'
  AND (effective_to IS NULL OR effective_to >= DATE'2024-04-01')
ORDER BY role
""").collect()
got = sorted((r.customer_key, r.role) for r in asof)
print("A1 membership as of 2024-04-01:", got)
assert got == [("C1", "owner"), ("C3", "signatory")], got

# (3) invariant: no overlapping intervals per (customer, account, role)
viol = spark.sql("""
WITH seq AS (
  SELECT customer_key, account_key, role, effective_to,
         LEAD(effective_from) OVER (
           PARTITION BY customer_key, account_key, role ORDER BY effective_from) AS next_from
  FROM bridge
)
SELECT COUNT(*) AS overlaps FROM seq
WHERE next_from IS NOT NULL AND effective_to IS NOT NULL AND next_from <= effective_to
""").first().overlaps
assert viol == 0

# (4) C1's full point-in-time footprint
foot = sorted((r.account_key, r.role) for r in spark.sql("""
  SELECT account_key, role FROM bridge WHERE customer_key = 'C1' ORDER BY account_key
""").collect())
assert foot == [("A1", "owner"), ("A2", "beneficiary")]
print("Q12 verified: as-of query, zero-overlap invariant, and footprint all correct.")
print("Talk track: fact joins use fact_date BETWEEN effective_from AND effective_to (or")
print("COALESCE(to, '9999-12-31')) — same predicate shape as an SCD2 join; late membership")
print("corrections = close the old interval and insert a corrected one, never UPDATE history.")

## Q13 — Data Modeling: Lakehouse Platform Design — End-to-End [HARD]

**Scenario.** A 100-team company is standardizing on one data platform. You own the reference architecture. Design it end-to-end and defend each choice — this is a synthesis question; everything from Days 15–21 feeds into it.

**Design areas**
1. **Ingestion** — how does each source class get in: file drops and event streams (Auto Loader / streaming tables), CDC from operational DBs (Lakeflow Connect, CDF, or Flink-style), SaaS connectors, and what lands in bronze.
2. **Medallion layering** — what bronze/silver/gold actually guarantee; when skipping silver is legitimate; who owns each layer's contract.
3. **Governance** — Unity Catalog layout (catalog per domain, schemas per product), isolation modes, row filters/column masks, lineage, and where data contracts are enforced.
4. **Serving** — batch BI, streaming, ML feature pipelines, and AI/GenAI (vector search, model serving) off the same gold tables.
5. **Cost & DR** — built into the design, not bolted on.

**Deliverable:** the architecture as a decision table (source → pattern → layer), plus the three hardest trade-offs you'd flag to the CTO.

In [0]:
# Solution Q13 — talk track: the reference architecture as decision tables

rows = [
    ("source class", "pattern", "lands in"),
    ("file drops (S3/ADLS/GCS)", "Auto Loader / streaming table into bronze", "bronze (raw, append)"),
    ("event streams (Kafka/Kinesis)", "Structured Streaming + Delta, exactly-once via idempotent sinks", "bronze (raw) -> silver (cleansed)"),
    ("operational DB CDC", "Lakeflow Connect / CDF consumer (Day 21 Q5!)", "silver (conformed)"),
    ("SaaS (Salesforce, Workday...)", "managed connectors, schema-registry-gated (Day 21 Q9)", "bronze/silver"),
    ("gold serving", "dimensional marts + feature tables + vector indexes", "gold (product-owned)"),
]
print("DECISION TABLE")
for r in rows:
    print("  | " + " | ".join(r))

print("""LAYER CONTRACTS
  bronze: immutable, schema-on-read, full history, replayable; owned by platform
  silver: cleaned + conformed + data contracts enforced (types, nullability, keys);
         THE layer consumers trust; skipping silver is legit only for teams whose
         gold consumers can tolerate raw quirks (sandboxes, feature exploration)
  gold:   product/domain-owned serving assets: star schemas, wide tables, feature
         tables, vector indexes; SLAs are defined HERE, not lower""")

print("""THE THREE TRADE-OFFS TO FLAG TO THE CTO
  1. Central platform team vs domain ownership: enforce contracts centrally,
     let domains own gold — full data mesh fails under 15-20 teams, full central
     bottlenecks past ~50. Pick federated-with-guardrails (UC + policies).
  2. Streaming everywhere vs batch-where-possible: streaming is a maintenance
     tax (state, watermarks, exactly-once, Day 19-21 territory). Stream what has
     a real freshness SLA; batch the rest nightly.
  3. Open formats vs velocity: open Delta/Iceberg guarantees no lock-in but
     caps you to format-speed evolution; proprietary acceleration (Photon,
     deletion vectors) is fine as long as the TABLE stays open.""")

print("""GOVERNANCE & COST/DR ONE-LINERS
  - catalog per domain, schema per product; shared mode for ad-hoc, isolated for prod
  - row filters + column masks at the TABLE, not in every query (Day 20 Q12)
  - unit economics from Day 21 Q18 as the cost lens; DR = replicated catalogs +
    DEEP CLONE gold + replayable bronze (Day 19 Q10) — tested quarterly, not documented""")
print("Q13 complete — talk track, no execution needed.")

## Q14 — Snowflake: Openflow — Managed Ingestion into the Snowflake Lakehouse [ADVANCED]

**Scenario.** Day 13 covered the classic Snowflake load paths (`COPY INTO`, Snowpipe, Snowpipe Streaming). Since 2025 Snowflake's answer to Lakeflow Connect is **Openflow**: managed, connector-based ingestion from SaaS apps, operational databases, and event sources, writing into Snowflake-managed tables and Iceberg tables. You're asked how it changes the ELT architecture and when to pick what.

**Tasks**
1. Place Openflow in the ingestion decision tree: managed connectors vs `COPY INTO` files vs Snowpipe streaming vs external tables.
2. Explain what "managed" buys you (no orchestration of loads, schema drift handling, CDC from sources) and what it costs (source coverage limits, egress/latency, preview-to-GA churn — verify the connector list before committing).
3. The cross-platform interview angle: compare against Databricks Lakeflow Connect ingesting into Delta — same problem class, and what actually differentiates the two is the TARGET format and governance model, not the connector idea.
4. Talk track: where Openflow lands data (managed vs Iceberg tables), why Iceberg-as-target matters for multi-engine access, and the licensing/pricing angle to ask your SE.

In [0]:
# Solution Q14 — talk track: the 2025/26 ingestion decision tree (illustrative, not executed)

rows = [
    ("situation", "pick", "why"),
    ("SaaS / operational DB CDC, no infra team", "Openflow managed connector",
     "managed scheduling, schema drift, no glue code to own"),
    ("partner drops files in a stage", "COPY INTO (batch)",
     "files arrive on a cadence; COPY is cheap, simple, exactly-once by file"),
    ("sub-minute event streams", "Snowpipe Streaming / Kafka connector",
     "row-level ingestion, low latency; Snowpipe batch is the fallback"),
    ("huge external lake, query in place", "external / Iceberg tables",
     "no copy; but no Snowflake-managed clustering/compaction either"),
]
for r in rows:
    print("  | " + " | ".join(r))

print("""WHAT 'MANAGED' BUYS AND COSTS
  + : no load orchestration to build, connector upgrades handled, CDC semantics
      (upserts/deletes) handled per source, schema drift surfaced not swallowed
  - : source coverage is finite (check current GA list), per-source pricing, and
      fast-moving feature surface — pin your design to GA capabilities only
  target matters: Openflow can land into managed tables OR Iceberg tables;
      Iceberg-as-target = multi-engine read (Spark/Trino/Databricks) on the same bytes
""")

print("""VS LAKEFLOW CONNECT (DATABRICKS)
  same problem class, different target: Openflow -> Snowflake tables/Iceberg;
  Lakeflow Connect -> Delta in UC. Differentiators that actually matter:
    1. target format + governance model (UC vs Snowflake catalog) - pick your
       governance plane first, connector second
    2. transformation plane: both push ELT down to their SQL engine
    3. both are 'managed ingestion' -- so the honest interview answer is about
       where you want your DATA and POLICIES to live, not which wizard is nicer
  (Facts as of late 2026 -- re-verify GA status of specific connectors in the docs.)""")
print("Q14 complete — talk track, no execution needed.")

## Q15 — Snowflake: Adaptive Warehouses & Sizing Economics [ADVANCED]

**Scenario.** Day 10 covered multi-cluster scaling; the sizing question itself is still open. Snowflake's 2025 evolution is the **adaptive warehouse** — a warehouse that automatically resizes with the workload instead of waiting for an operator to notice. You're asked both for the decision framework and for the credit math that decides when scaling up actually pays.

**Tasks**
1. State the size ladder and the credit math: XS=1, S=2, M=4, L=8, XL=16 … 3XL=64 credits/hour — each size step doubles credits AND (roughly) halves query time only if the query is parallelizable.
2. Run the sizing exercise: a query takes 40 min on XS, 10 min on M, 8 min on L. Compute credits for each and decide.
3. Explain what adaptive warehouses change operationally (less manual resizing, but less predictable spend — pair with resource monitors) and what they cannot fix (data-model problems, single-threaded query sections).
4. Talk track: auto-suspend/auto-resume economics, queues as the real signal to scale out (multi-cluster) vs scale up, and why "fastest size" is almost never the cheapest per query.

In [0]:
# Solution Q15 — the credit math (sizes: XS=1, S=2, M=4, L=8 credits/hr)

sizes = {"XS": 1, "S": 2, "M": 4, "L": 8, "XL": 16, "2XL": 32, "3XL": 64}
runtimes_min = {"XS": 40, "M": 10, "L": 8}          # measured on the workload

credits = {s: sizes[s] * (m / 60) for s, m in runtimes_min.items()}
for s, c in credits.items():
    print(f"{s:>3}: {runtimes_min[s]:>3} min -> {c:.2f} credits")

# scale-up is often cost-NEUTRAL: XS and M cost the same, M is 4x faster
assert abs(credits["XS"] - credits["M"]) < 1e-9
# ...but going TOO far costs more: L burns extra credits for marginal time
assert credits["L"] > credits["M"]
print("Verdict: pick M — same credits as XS, 4x faster. L pays a premium for 2 more minutes.")

print("""DECISION FRAMEWORK (say this in the room)
  1. start small (XS/S), measure, size up one step at a time
  2. scale UP when a single query is slow and parallelizable (credits often flat)
  3. scale OUT (multi-cluster) when QUEUES form - concurrency, not query speed
  4. adaptive warehouses remove the manual step 2 - but pair with resource
     monitors because adaptivity optimizes latency, not your bill
  5. fastest size is rarely cheapest per query: beyond the parallelizable
     region, extra size buys minutes and costs doubling credits
  6. auto-suspend + auto-resume only if gaps between queries exceed resume
     overhead (~60s spin-up); constant trickle loads keep warehouses hot
""")
print("Q15 verified: cost-neutral scale-up + premium overshoot both proven numerically.")

## Q16 — Monitoring: Alert Fatigue — Precision / Recall for Monitoring Rules [ADVANCED]

**Scenario.** Your on-call rotation gets 40 alert notifications a day and ignores most of them. Treat alert rules like a **classifier** and evaluate them with the same rigor as an ML model: a rule that fires on 30 days of history can be scored for precision, recall, and false-positive rate — then tuned and de-duplicated.

**Tasks**
1. Simulate 30 days: 5 real outage days (freshness lag 220–480 min), 21 quiet days (lag 8–58), and 4 noisy-but-benign days (cache-cold starts, lag 105–160).
2. Implement `evaluate(threshold)` returning alerts/TP/FP/FN plus precision, recall, and false-positive rate per day.
3. Show the loose threshold (60 min): 9 alerts, precision ≈ 0.56 — then tune to 200 min and get precision 1.0 with **recall unchanged**. Explain when tuning like this is safe (separated distributions) and when it is not (overlapping distributions — you must then trade recall for precision explicitly).
4. Grouping: 24 raw hourly firings collapse into 1 incident notification (24× reduction). Talk track: maintenance windows, flapping suppression, symptom-vs-cause paging, severity routing.

In [0]:
# Solution Q16 — score alert rules like a classifier, then tune + deduplicate
import random

days = list(range(1, 31))
true_outages = {4, 9, 17, 22, 28}
lag = {d: (random.Random(d).uniform(220, 480) if d in true_outages
           else random.Random(1000 + d).uniform(8, 58)) for d in days}
for d in (6, 13, 25, 30):                       # noisy-but-benign cache-cold days
    lag[d] = random.Random(2000 + d).uniform(105, 160)

def evaluate(threshold):
    alerts = {d for d in days if lag[d] > threshold}
    tp = len(alerts & true_outages)
    fp = len(alerts - true_outages)
    fn = len(true_outages - alerts)
    prec = tp / (tp + fp) if tp + fp else 1.0
    return {"alerts": len(alerts), "tp": tp, "fp": fp, "fn": fn,
            "precision": prec, "recall": tp / (tp + fn), "fpr_per_day": fp / len(days)}

loose, tight = evaluate(60), evaluate(200)
print("threshold  60:", {k: round(v, 3) if isinstance(v, float) else v for k, v in loose.items()})
print("threshold 200:", {k: round(v, 3) if isinstance(v, float) else v for k, v in tight.items()})
assert loose["fp"] == 4 and loose["fn"] == 0
assert abs(loose["precision"] - 5 / 9) < 1e-9 and loose["recall"] == 1.0
assert tight["fp"] == 0 and tight["fn"] == 0
assert tight["precision"] == 1.0 and tight["recall"] == 1.0

raw = 24 * loose["alerts"]                      # hourly rule firing raw
incidents = loose["tp"] + loose["fp"]           # grouped incident notifications
print(f"raw hourly alerts: {raw} -> incident notifications: {incidents} ({raw // incidents}x cut)")
assert raw / incidents == 24
print("Q16 verified: precision 0.556 -> 1.0 at same recall; grouping cuts volume 24x.")
print("Caveat to say out loud: tuning worked because the distributions are separated;")
print("with overlap you MUST choose a recall/precision point — say which pager you")
print("are willing to miss (freshness = page on symptom, ticket on cause).")

## Q17 — Monitoring: Correlated Observability — One Run, One Story [ADVANCED]

**Scenario.** At 07:00 someone asks: *what happened in the 03:00 nightly run?* The answer is currently spread across job logs, notebook stdout, warehouse query history, and three people's memories. Design **correlated observability**: a single `run_id` threads through every log line, every query, and every artifact of a run, so any engineer can reconstruct the whole story in one query.

**Tasks**
1. Implement the logging core: a `logging.Filter` that injects the `run_id` into every record, plus a JSON handler that captures structured events (extract → transform → load, with row counts, quarantined counts, durations, SLA warnings). Assert every captured record carries the run_id.
2. Show the one-run reconstruction: replay the captured records as a timeline.
3. Talk track (illustrative snippets in the solution): tagging Spark/warehouse queries (`spark.conf` / `SET` query tags) so `system.query.history` filters by run; capturing job + task run IDs as metadata; the drill — pick last Tuesday's failed run and reconstruct it end-to-end as an on-call exercise.
4. What to log, what not to: start/end + row counts + durations + errors **with context** (which slice, which file, which key range) — never raw payloads, never PII, and never `print`.

In [0]:
# Solution Q17 — run_id threading through structured logs + reconstruction drill
import logging, time

class RunContextFilter(logging.Filter):
    def __init__(self, run_id):
        super().__init__()
        self.run_id = run_id
    def filter(self, record):
        record.run_id = self.run_id        # inject into EVERY record
        return True

class JSONListHandler(logging.Handler):
    def __init__(self):
        super().__init__()
        self.records = []
    def emit(self, record):
        self.records.append({"ts": record.created, "level": record.levelname,
                             "logger": record.name, "msg": record.getMessage(),
                             "run_id": record.run_id})

logger = logging.getLogger("pipeline.d21")
logger.setLevel(logging.INFO)
h = JSONListHandler()
logger.addHandler(h)
logger.addFilter(RunContextFilter("run-2026-11-03-4f2a"))

steps = ["extract: 1,204,553 rows read from bronze.orders",
         "transform: 12 rows quarantined (schema drift: amt STRING)",
         "load: MERGE into gold.orders complete in 41s"]
for s in steps:
    time.sleep(0.01)
    logger.info(s)
logger.warning("sla: silver freshness lag 25m approaching 30m budget")

assert len(h.records) == 4
assert all(r["run_id"] == "run-2026-11-03-4f2a" for r in h.records)
timeline = [f"[{r['level']:7s}] {r['msg']}" for r in h.records]
print("\n".join(timeline))
assert timeline[0].startswith("[INFO   ] extract")
assert any(r["level"] == "WARNING" for r in h.records)
print("Q17 verified: every line carries the run_id; the whole run reconstructs from the handler.")

# Illustrative (not executed): tagging queries so system.query.history finds them
snippet = (
    "spark.conf.set('spark.runId', run_id)          # lands in query history/tags\n"
    "-- SQL warehouse: SET spark.queryId = run_id;  # filter system.query.history on it\n"
    "SELECT statement_text, total_time_ms, start_time FROM system.query.history\n"
    "WHERE (statement_text LIKE '%run-2026-11-03-4f2a%' OR get(tags, 'runId') = 'run-2026-11-03-4f2a')\n"
    "  AND start_time BETWEEN '2026-11-03 03:00' AND '2026-11-03 04:00'"
)
print("---- illustrative query-tagging pattern ----")
print(snippet)

## Q18 — Monitoring: From Attribution to Unit Economics + Burn Anomaly Detection [HARD]

**Scenario.** Day 9 and Day 13 built cost **attribution** (who spent what via `system.billing.usage`). The finance lead's next question is harder: *what does one unit of data product cost, and is today's burn normal?* That's **unit economics** — cost divided by output — plus anomaly detection on the spend stream.

**Tasks**
1. Build unit-cost KPIs from run records: dollars per 1M rows processed, per pipeline, and roll up to a **showback** table by team (showback educates, chargeback punishes — know the difference).
2. Implement robust **burn anomaly detection**: median + MAD-based robust z-score (why not mean/std? spend is skewed — one big day wrecks a mean-based baseline and hides the very spikes you hunt). A day with a runaway auto-scale event (≈3× normal) must be flagged with `z > 3.5`.
3. Assert the injected anomaly day is the only flagged day, and that the expensive team is visible in the showback KPIs.
4. Talk track: choosing the denominator (rows? runs? business events? — pick the unit your consumer cares about), what happens after the anomaly fires (freeze + RCA + cap, not just a Slack message), and the maturity ladder attribution → showback → chargeback → unit-economics-driven design.

In [0]:
# Solution Q18 — unit KPIs + robust (median/MAD) burn anomaly detection
import statistics, random

runs = []
rng = random.Random(42)
for d in range(1, 31):
    rows = 1_000_000 + int(rng.uniform(-50_000, 50_000))
    dollars = 4.0 + rng.uniform(-0.6, 0.6)          # ~$4 per daily run
    if d == 24:
        dollars = 11.5                              # runaway auto-scale day
    runs.append({"day": d, "pipeline": "gold_orders", "rows": rows, "dollars": round(dollars, 2)})

costs = [r["dollars"] for r in runs]
med = statistics.median(costs)
mad = statistics.median([abs(c - med) for c in costs])
robust_z = {r["day"]: 0.6745 * (r["dollars"] - med) / mad for r in runs}
flagged = [d for d, z in sorted(robust_z.items()) if z > 3.5]
print(f"median run cost = ${med:.2f}  |  robust-z flags: {flagged}")
assert flagged == [24], flagged

unit = {r["day"]: r["dollars"] * 1_000_000 / r["rows"] for r in runs}   # $ per 1M rows
assert unit[24] > 3 * (med / 1.05)                 # unit cost spiked > 3x

teams = {"orders": {"runs": 200, "dollars": 810.0, "rows": 210_000_000},
         "risk":   {"runs": 90,  "dollars": 355.0, "rows":  40_000_000}}
showback = {t: round(v["dollars"] * 1_000_000 / v["rows"], 4) for t, v in teams.items()}
print("showback — $ per 1M rows by team:", showback)
assert showback["risk"] > showback["orders"]
print("Q18 verified: MAD-based detector caught the runaway day alone; the expensive")
print("team is visible in unit KPIs. Talk: median/MAD because spend is skewed; pick")
print("the denominator your CONSUMER cares about; after the alert: freeze, RCA, cap —")
print("a Slack message is not a remediation. Ladder: attribution -> showback -> chargeback.")

# Day 21 — Score Card

Grade each question 0–3: **0** couldn't start · **1** needed hints · **2** solved with bugs · **3** clean under time.

| Q# | Category | Topic | Difficulty | Score | Notes |
|---|----------|-------|-----------|-------|-------|
| Q1 | SQL Advanced | Markov transition matrix | ADVANCED | | |
| Q2 | SQL Advanced | Connected components (label propagation) | HARD | | |
| Q3 | Python | RAG retrieval metrics | ADVANCED | | |
| Q4 | Python | Bloom filter from scratch | HARD | | |
| Q5 | PySpark | Delta CDF consumer | ADVANCED | | |
| Q6 | PySpark | SCD2 invariants auditor | ADVANCED | | |
| Q7 | Spark Optimization | Deletion vectors | ADVANCED | | |
| Q8 | Spark Optimization | Input split sizing & scan parallelism | ADVANCED | | |
| Q9 | ETL | Schema-registry compatibility checker | ADVANCED | | |
| Q10 | ETL | REPLACE WHERE scoped reload | ADVANCED | | |
| Q11 | ETL | Partial-batch isolation & SLA-aware loads | ADVANCED | | |
| Q12 | Data Modeling | Time-scoped bridge table | ADVANCED | | |
| Q13 | Data Modeling | Lakehouse platform design — end-to-end | HARD | | |
| Q14 | Snowflake | Openflow managed ingestion | ADVANCED | | |
| Q15 | Snowflake | Adaptive warehouses & sizing economics | ADVANCED | | |
| Q16 | Monitoring | Alert fatigue — precision/recall | ADVANCED | | |
| Q17 | Monitoring | Correlated observability | ADVANCED | | |
| Q18 | Monitoring | Unit economics + burn anomaly detection | HARD | | |

**Total: ___ / 54** — Week 3 complete. Day 22 starts Week 4: FAANG-level questions begin.